# 트리 모델의 손실 설명

모델 손실을 설명하는 것은 디버깅 및 모델 모니터링에 매우 유용할 수 있습니다. 이 노트북은 이것이 어떻게 작동하는지에 대한 매우 간단한 예를 제공합니다. 모델의 손실을 설명하려면 레이블을 전달해야 하며 TreeExplainer의 `feature_perturbation="independent"` 옵션에 대해서만 지원됩니다.

이 노트는 이 방법에 대한 전체 글을 게시한 후에 구체화될 것입니다.

In [1]:
import numpy as np
import xgboost

import shap

### XGBoost 분류기 학습

In [2]:
X, y = shap.datasets.adult()

model = xgboost.XGBClassifier()
model.fit(X, y)

# compute the logistic log-loss
model_loss = -np.log(model.predict_proba(X)[:, 1]) * y + -np.log(model.predict_proba(X)[:, 0]) * (1 - y)

model_loss[:10]

array([8.43880873e-04, 2.47898608e-01, 1.17997164e-02, 7.11527169e-02,
       6.41849875e-01, 1.76084566e+00, 5.70287136e-03, 8.60033274e-01,
       4.78262809e-04, 6.43801317e-03])

### TreeExplainer를 사용하여 모델의 로그 손실 설명

모델 손실의 '예상_값'은 라벨에 따라 달라지므로 이제 단일 숫자가 아닌 함수입니다.

In [3]:
explainer = shap.TreeExplainer(model, X, feature_perturbation="interventional", model_output="log_loss")
explainer.shap_values(X.iloc[:10, :], y[:10]).sum(1) + np.array([explainer.expected_value(v) for v in y[:10]])

array([8.43887488e-04, 2.47898585e-01, 1.17997435e-02, 7.11527711e-02,
       6.41849874e-01, 1.76084475e+00, 5.70285151e-03, 8.60033255e-01,
       4.78233521e-04, 6.43796897e-03])